In [3]:
import ipywidgets as widgets
from IPython.display import display
import board
import busio
from adafruit_pca9685 import PCA9685
from adafruit_servokit import ServoKit
import time

def i2c_scan(i2c):
    while not i2c.try_lock():
        pass
    try:
        devices = i2c.scan()
        return devices
    finally:
        i2c.unlock()

i2c_bus = busio.I2C(board.SCL, board.SDA)
print("Scanning I2C bus...")
devices = i2c_scan(i2c_bus)
print(f"I2C devices found: {[hex(device) for device in devices]}")

pca = PCA9685(i2c_bus, address=0x40)
pca.frequency = 60  # PCA9685 주파수 설정

kit = ServoKit(channels=16, i2c=i2c_bus, address=0x60)

# =========================================================
# DC 모터 드라이버 클래스
# =========================================================
class PWMThrottleHat:
    def __init__(self, pwm, channel):
        self.pwm = pwm
        self.channel = channel

    def set_throttle(self, throttle):
        pulse = int(0xFFFF * abs(throttle))
        
        if throttle < 0:
            self.pwm.channels[self.channel + 5].duty_cycle = pulse
            self.pwm.channels[self.channel + 4].duty_cycle = 0
            self.pwm.channels[self.channel + 3].duty_cycle = 0xFFFF
        elif throttle > 0:
            self.pwm.channels[self.channel + 5].duty_cycle = pulse
            self.pwm.channels[self.channel + 4].duty_cycle = 0xFFFF
            self.pwm.channels[self.channel + 3].duty_cycle = 0
        else:
            self.pwm.channels[self.channel + 5].duty_cycle = 0
            self.pwm.channels[self.channel + 4].duty_cycle = 0
            self.pwm.channels[self.channel + 3].duty_cycle = 0

motor_hat = PWMThrottleHat(pca, channel=0)

# =========================================================
# 상태 변수 설정
# =========================================================
pan_init = 98
pan_angle = pan_init       # 초기 서보 각도 (0~180)
current_speed = 0.5   # 기본 속도 (50%)
kit.servo[0].angle = pan_angle

# =========================================================
# ipywidgets UI 요소 생성
# =========================================================
# 버튼 스타일 정의
btn_layout = widgets.Layout(width='110px', height='45px')

btn_w = widgets.Button(description='▲ 전진', button_style='primary', layout=btn_layout)
btn_s = widgets.Button(description='▼ 후진', button_style='primary', layout=btn_layout)
btn_a = widgets.Button(description='◀ 좌회전', button_style='info', layout=btn_layout)
btn_d = widgets.Button(description='우회전 ▶', button_style='info', layout=btn_layout)
btn_stop = widgets.Button(description='⏹ 정지', button_style='danger', layout=btn_layout)
btn_center = widgets.Button(description='🎯 중앙정렬', button_style='warning', layout=btn_layout)

# 속도 설정 슬라이더
speed_slider = widgets.FloatSlider(
    value=0.5, min=0.1, max=1.0, step=0.05,
    description='속도 조절:',
    continuous_update=True,
    orientation='horizontal'
)

# 상태 로그 출력 창
out_log = widgets.Output(layout=widgets.Layout(height='80px', overflow_y='auto', border='1px solid #ccc'))

# =========================================================
# 5. 이벤트 핸들러 함수 정의
# =========================================================
def log_msg(msg):
    with out_log:
        out_log.clear_output(wait=True)
        print(msg)

def on_w_click(b):
    motor_hat.set_throttle(speed_slider.value)
    log_msg(f"🚗 전진 중... (속도: {int(speed_slider.value*100)}%)")

def on_s_click(b):
    motor_hat.set_throttle(-speed_slider.value)
    log_msg(f"🚗 후진 중... (속도: {int(speed_slider.value*100)}%)")

def on_a_click(b):
    global pan_angle
    pan_angle = max(0, pan_angle - 10)
    kit.servo[0].angle = pan_angle
    log_msg(f"👈 좌회전 (서보 각도: {pan_angle}°)")

def on_d_click(b):
    global pan_angle
    pan_angle = min(180, pan_angle + 10)
    kit.servo[0].angle = pan_angle
    log_msg(f"👉 우회전 (서보 각도: {pan_angle}°)")

def on_stop_click(b):
    motor_hat.set_throttle(0)
    log_msg("🛑 모터 정지!")

def on_center_click(b):
    global pan_angle
    global pan_init
    pan_angle = pan_init
    kit.servo[0].angle = pan_angle
    log_msg(f"🎯 조향 중앙 정렬 완료 (서보 각도: {pan_angle}°)")

# 이벤트 연결
btn_w.on_click(on_w_click)
btn_s.on_click(on_s_click)
btn_a.on_click(on_a_click)
btn_d.on_click(on_d_click)
btn_stop.on_click(on_stop_click)
btn_center.on_click(on_center_click)

# =========================================================
# 6. UI 레이아웃 배치 및 출력
# =========================================================
row1 = widgets.HBox([widgets.Label(layout=widgets.Layout(width='110px')), btn_w, btn_center])
row2 = widgets.HBox([btn_a, btn_stop, btn_d])
row3 = widgets.HBox([widgets.Label(layout=widgets.Layout(width='110px')), btn_s])

control_panel = widgets.VBox([
    widgets.HTML("<h3>🏎️ Jetson Orin Nano RC카 제어 패널</h3>"),
    speed_slider,
    widgets.HTML("<hr>"),
    row1, row2, row3,
    widgets.HTML("<hr>"),
    widgets.Label("📋 실시간 동작 상태:"),
    out_log
])

display(control_panel)
log_msg("READY: 컨트롤 패널 준비 완료.")

Scanning I2C bus...
I2C devices found: ['0x40', '0x60']
